# Customer Churn Prediction

Predict whether a telecom customer will leave (`Churn = Yes`).

- **Target:** Churn (Yes/No) → binary classification
- **Important metrics:** Recall & F1 for Churn (missing a churner is costly)
- **Business use:** flag high-risk customers early and offer better plan / longer contract


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid")
%matplotlib inline


# Task 1 — Data Preparation

Load the data and check structure, target, missing values and duplicates.

In [ ]:
file_id = '1WaMMcWZQbYIQXBVOwJZ0tSjDgw5PsXyR'
url = f'https://drive.google.com/uc?id={file_id}'

df = pd.read_csv(url)
rows, columns = df.shape

print(f'Data Frame Rows = {rows}')
print(f'Data Frame Columns = {columns}')
df.head()


In [ ]:
df.info()


In [ ]:
df['Churn'].value_counts()


In [ ]:
# TotalCharges is text → convert to numbers (blanks become NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(df['TotalCharges'].dtype)


In [ ]:
print('Missing values %:')
print((df.isnull().mean().sort_values(ascending=False) * 100).head())
print()
print(f'Duplicate rows = {df.duplicated().sum()}')


In [ ]:
df.describe().T


In [ ]:
df.describe(include='object')


**Data notes**
1. 7043 rows, 21 columns
2. `TotalCharges` had blank strings → converted to numeric (few NaNs)
3. No duplicate rows
4. `customerID` is only an ID → drop before modelling
5. Class imbalance: more No than Yes → Accuracy alone is not enough


# Task 2 — Data Cleaning

Drop `customerID`, encode target.  
Missing `TotalCharges` will be filled **inside the pipeline** (median from training only) to avoid leakage.


In [ ]:
df.drop('customerID', axis=1, inplace=True)

df['Churn'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

df.head()


# Task 3 — Data Visualization

`Churn = 1` means left, `Churn = 0` means stayed.


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Churn')
plt.title('Target Distribution')
plt.show()


Most customers stay → target is imbalanced. That is why we care about Recall/F1, not only Accuracy.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='Contract', hue='Churn')
plt.title('Churn by Contract Type')
plt.show()

print(df.groupby('Contract')['Churn'].mean().mul(100).round(1).rename('Churn %'))


Month-to-month contracts have the highest churn rate. Longer contracts churn much less.

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='Churn', y='tenure')
plt.title('Tenure vs Churn')
plt.show()

print(df.groupby('Churn')['tenure'].describe()[['mean', '50%']].round(1))


Customers who leave usually have shorter tenure (newer customers churn more).

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges')
plt.title('Monthly Charges vs Churn')
plt.show()

print(df.groupby('Churn')['MonthlyCharges'].describe()[['mean', '50%']].round(1))


Customers who leave tend to pay higher monthly charges.

In [ ]:
plt.figure(figsize=(8, 5))
corr = df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation — Numeric Features')
plt.show()


# Task 4 — Train/Test Split & Preprocessing

- 80% train / 20% test  
- `stratify=y` to keep the same churn rate in both sets  
- Scaling + OneHot inside Pipeline (fit on train only)


In [ ]:
x = df.drop(columns=['Churn'])
y = df['Churn']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42, stratify=y
)

print('X_train:', x_train.shape)
print('X_test :', x_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)


In [ ]:
numerical_features = x_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = x_train.select_dtypes(include=['object']).columns.tolist()

print('Numerical Features:', numerical_features)
print('Categorical Features:', categorical_features)

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


# Task 5 — Model Comparison

Train **3 models** with the same pipeline and same split.  
Positive class = Churn (1).


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
}

test_results = []
train_results = []

for name, model in models.items():
    print(name)

    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    model_pipeline.fit(x_train, y_train)
    y_predict_train = model_pipeline.predict(x_train)
    y_predict_test = model_pipeline.predict(x_test)

    # Training metrics
    train_f1 = f1_score(y_train, y_predict_train, pos_label=1)
    train_recall = recall_score(y_train, y_predict_train, pos_label=1)
    train_precision = precision_score(y_train, y_predict_train, pos_label=1)
    train_accuracy = accuracy_score(y_train, y_predict_train)

    # Testing metrics
    test_f1 = f1_score(y_test, y_predict_test, pos_label=1)
    test_recall = recall_score(y_test, y_predict_test, pos_label=1)
    test_precision = precision_score(y_test, y_predict_test, pos_label=1)
    test_accuracy = accuracy_score(y_test, y_predict_test)

    print('--- Training Data Classification Report ---')
    print(classification_report(
        y_train, y_predict_train,
        labels=[0, 1], target_names=['No Churn', 'Churn'], zero_division=0
    ))

    print('--- Test Data Classification Report ---')
    print(classification_report(
        y_test, y_predict_test,
        labels=[0, 1], target_names=['No Churn', 'Churn'], zero_division=0
    ))

    train_results.append({
        'Model': name,
        'Train F1-Score': train_f1,
        'Train Recall': train_recall,
        'Train Precision': train_precision,
        'Train Accuracy': train_accuracy
    })
    test_results.append({
        'Model': name,
        'Test F1-Score': test_f1,
        'Test Recall': test_recall,
        'Test Precision': test_precision,
        'Test Accuracy': test_accuracy
    })
    print('\n ===========================================\n')

train_df = pd.DataFrame(train_results).set_index('Model').round(3)
test_df = pd.DataFrame(test_results).set_index('Model').round(3)


In [ ]:
train_df.T


In [ ]:
test_df.T


**Observations**
- Decision Tree & Random Forest look almost perfect on **train** but drop on **test** → overfitting
- Logistic Regression is more stable between train and test
- Next: tune one model with GridSearchCV


# Task 6 — Tuning (Gradient Boosting)

GridSearchCV on training data only (scoring = F1).  
Test set is used **once** at the end.


In [ ]:
gb_model = GradientBoostingClassifier(random_state=42)

param_grid = {
    'classifier__n_estimators': [50, 100, 150],
    'classifier__learning_rate': [0.05, 0.1, 0.2],
    'classifier__max_depth': [2, 3, 4],
}

T_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', gb_model)
])

grid_search = GridSearchCV(
    T_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1
)
grid_search.fit(x_train, y_train)

print('Best Parameters:', grid_search.best_params_)
print('Best CV F1:', round(grid_search.best_score_, 3))

best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(x_test)

print('Classification report (test)')
print(classification_report(y_test, y_pred_tuned, target_names=['No Churn', 'Churn']))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_tuned, display_labels=['No Churn', 'Churn']
)
plt.title('Confusion Matrix — Tuned Gradient Boosting')
plt.show()


# Task 6 — Explain Results and Business Value

### 26. Confusion Matrix (final model)

The confusion matrix above is for the **tuned Gradient Boosting** model on the held-out test set.

| Term | Meaning in this project |
|------|-------------------------|
| **TP (True Positive)** | Customer was going to leave, and the model correctly predicted Churn → retention team can contact them |
| **FP (False Positive)** | Customer was going to stay, but the model predicted Churn → false alarm (wasted contact, usually cheap) |
| **TN (True Negative)** | Customer was going to stay, and the model correctly predicted No Churn → left alone |
| **FN (False Negative)** | Customer was going to leave, but the model predicted No Churn → **missed churner (most costly error)** |

That is why **Recall** for the Churn class matters: we want to catch as many real churners as possible (reduce FN).


### 27. Two important patterns (correlation, not causation)

1. **Short tenure + Month-to-month contract**  
   Customers who have been with the company for a short time and are still on a month-to-month plan show a much higher churn rate.  
   This is a pattern in the data — it does **not** prove that the contract type alone *causes* them to leave.

2. **Higher MonthlyCharges**  
   Customers who leave tend to pay more per month than customers who stay.  
   Again, this is association only: higher charges may reflect richer service packages, not the reason they churn.


### 28. Two actions the business can take

1. **Contact high-risk customers early** with a personalized offer (discount, loyalty bonus, or free add-on) before they decide to leave.
2. **Encourage a longer contract** (one-year or two-year) with a small incentive — longer contracts are linked to much lower churn in the data.


### 29. At least two limitations

1. **Class imbalance** — only about 27% of customers churn. The model can still miss a non-trivial share of real churners (False Negatives), and Accuracy alone looks better than it is.
2. **Data is a snapshot** — customer behaviour and market conditions change over time. The model should be retrained periodically; old patterns may not hold forever.
3. **False alarms (FP)** — contacting customers who were not going to leave costs time and money, even if less than losing a customer.


### 30. Final recommendation

I would choose the **tuned Gradient Boosting** model because:
- It was selected with cross-validation on the training set (not by peeking at the test set).
- It improves the balance between Precision and Recall for the Churn class compared with the untuned tree models.
- It clearly beats a naive “always predict No Churn” approach on Recall and F1.

If the business needs a very simple and interpretable model for managers, **Logistic Regression** is a solid second choice (stable train/test scores, easy to explain).


### Optional challenge — Prediction threshold

By default the model uses threshold **0.5**: if predicted probability of Churn ≥ 0.5 → predict Yes.  

Changing the threshold changes the trade-off:
- **Lower threshold** (e.g. 0.3) → higher Recall (catch more churners) but lower Precision (more false alarms).
- **Higher threshold** (e.g. 0.7) → higher Precision but lower Recall (miss more churners).

We explore this on the **training / validation side only** — we do **not** pick the threshold by repeatedly optimizing on the final test set.


In [ ]:
# Optional: effect of threshold on Recall & Precision
# Use predicted probabilities from the tuned model on the TEST set only for illustration
# (threshold choice itself should be guided by validation, not repeated test tuning)

y_proba = best_model.predict_proba(x_test)[:, 1]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
rows = []

for t in thresholds:
    y_hat = (y_proba >= t).astype(int)
    rows.append({
        'Threshold': t,
        'Precision': round(precision_score(y_test, y_hat, zero_division=0), 3),
        'Recall': round(recall_score(y_test, y_hat, zero_division=0), 3),
        'F1': round(f1_score(y_test, y_hat, zero_division=0), 3),
    })

threshold_df = pd.DataFrame(rows)
threshold_df


**How to read the threshold table**
- As the threshold goes **down**, Recall usually goes **up** (we catch more churners) and Precision goes **down** (more false alarms).
- As the threshold goes **up**, Precision improves but we miss more real churners.
- The business can pick a threshold based on cost of FN vs FP — without treating the final test set as a tuning playground.
